In [1]:
import sqlite3
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re 


In [2]:
import sqlite3
import requests
from bs4 import BeautifulSoup
import pandas as pd

def create_database(DB_PATH):
    """Create the database and the Conference table."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS Conference (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        Title TEXT NOT NULL,
        Author TEXT NOT NULL,
        PDF_Link TEXT,
        Code_URL TEXT,
        Conference_Name TEXT NOT NULL
    )
    ''')

    print("Conference 테이블이 생성되었습니다.")
    conn.commit()
    conn.close()

def save_to_database(df, conference_name, DB_PATH):
    conn = sqlite3.connect(DB_PATH, timeout=10)
    cursor = conn.cursor()

    try:
        for _, row in df.iterrows():  # ✅ iterrows() 사용하여 DataFrame의 각 행을 처리
            # 중복 데이터 확인
            cursor.execute('''
            SELECT 1 FROM Conference WHERE Title = ? AND Author = ? AND Conference_Name = ?
            ''', (row['title'], row['authors'], conference_name))
            result = cursor.fetchone()

            if not result:
                cursor.execute('''
                INSERT INTO Conference (Title, Author, PDF_Link, Code_URL, Conference_Name)
                VALUES (?, ?, ?, ?, ?)
                ''', (row['title'], row['authors'], row['pdf_link'], row['code_url'], conference_name))

        conn.commit()
        print(f"{len(df)}개의 논문이 {conference_name}에 저장되었습니다.")
    except sqlite3.Error as e:
        print(f"Database error: {e}")
    finally:
        conn.close()

In [20]:

def get_www_papers(html_file_path, conference_name):
    """저장된 HTML 파일에서 ICDM 학회의 Accepted Papers 정보를 크롤링 후 DataFrame 반환"""

    # HTML 파일 읽기
    with open(html_file_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")

    papers = []

    # 논문 리스트가 포함된 섹션 찾기
    for entry in soup.find_all("li", class_="entry inproceedings"):
        # 제목 찾기
        title_tag = entry.find("span", class_="title")
        title_text = title_tag.text.strip() if title_tag else "Unknown"

        # 저자 찾기
        authors_tags = entry.find_all("span", itemprop="author")
        authors_list = [author.find("span", itemprop="name").text.strip() for author in authors_tags]
        authors_cleaned = ", ".join(authors_list)

        # DOI 링크 찾기 (electronic edition via DOI)
        doi_tag = entry.find("a", href=re.compile(r"doi\.org"))
        if doi_tag and doi_tag.get("href"):
            pdf_link = doi_tag["href"]
        else:
            pdf_link = None

        # 논문 정보 추가
        papers.append({
            "title": title_text,
            "authors": authors_cleaned,
            "pdf_link": pdf_link,
            'code_url': None,
            "conference_name": conference_name
        })

    # DataFrame으로 변환
    df_papers = pd.DataFrame(papers)
    return df_papers

In [12]:
url = 'https://dblp.org/db/conf/icdm/icdm2024.html'
DB_PATH = "con_db/ICDM_conference_2024.db"
conference_name = 'ICDM 2024'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [13]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/ICDM_2024_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [23]:
df_papers = get_www_papers('html/ICDM_2024_accepted_papers.html', conference_name)

In [24]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Feature Map Purification for Enhancing Adversa...,"Mubarak G. Abdu-Aguye, Muhammad Zaigham Zaheer...",https://doi.org/10.1109/ICDM59182.2024.00007,None,ICDM 2024
1,Margin-Bounded Confidence Scores for Out-of-Di...,"Lakpa Dorje Tamang, Mohamed Reda Bouadjenek, R...",https://doi.org/10.1109/ICDM59182.2024.00053,None,ICDM 2024
2,LISA: Learning-Integrated Space Partitioning F...,"Bang An, Xun Zhou, Amin Vahedian, W. Nick Stre...",https://doi.org/10.1109/ICDM59182.2024.00008,None,ICDM 2024
3,Normalizing Self-Supervised Learning for Prova...,"Alexandra Bazarova, Evgenia Romanenkova, Alexe...",https://doi.org/10.1109/ICDM59182.2024.00009,None,ICDM 2024
4,Fast and Accurate Triangle Counting in Graph S...,"Cristian Boldrin, Fabio Vandin",https://doi.org/10.1109/ICDM59182.2024.00010,None,ICDM 2024


In [25]:
save_to_database(df_papers, conference_name, DB_PATH)

116개의 논문이 ICDM 2024에 저장되었습니다.


# 2023

In [26]:
url = 'https://dblp.org/db/conf/icdm/icdm2023.html'
DB_PATH = "con_db/ICDM_conference_2023.db"
conference_name = 'ICDM 2023'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [27]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/ICDM_2023_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [28]:
df_papers = get_www_papers('html/ICDM_2023_accepted_papers.html', conference_name)

In [29]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,A Deep Reinforcement Learning Approach to Conf...,"Amir Abolfazli, Jakob Spiegelberg, Gregory Pal...",https://doi.org/10.1109/ICDM58522.2023.00009,None,ICDM 2023
1,Reinforcement Neighborhood Selection for Unsup...,"Yuanchen Bei, Sheng Zhou, Qiaoyu Tan, Hao Xu, ...",https://doi.org/10.1109/ICDM58522.2023.00010,None,ICDM 2023
2,CAC: Enabling Customer-Centered Passenger-Seek...,"Palawat Busaranuvong, Xin Zhang, Yanhua Li, Xu...",https://doi.org/10.1109/ICDM58522.2023.00011,None,ICDM 2023
3,Graph Self-Contrast Representation Learning.,"Minjie Chen, Yao Cheng, Ye Wang, Xiang Li, Min...",https://doi.org/10.1109/ICDM58522.2023.00012,None,ICDM 2023
4,A Practical Clean-Label Backdoor Attack with L...,"Peng Chen, Jirui Yang, Junxiong Lin, Zhihui Lu...",https://doi.org/10.1109/ICDM58522.2023.00013,None,ICDM 2023


In [30]:
save_to_database(df_papers, conference_name, DB_PATH)

203개의 논문이 ICDM 2023에 저장되었습니다.


# 2022

In [31]:
url = 'https://dblp.org/db/conf/icdm/icdm2022.html'
DB_PATH = "con_db/ICDM_conference_2022.db"
conference_name = 'ICDM 2022'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [32]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/ICDM_2022_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [33]:
df_papers = get_www_papers('html/ICDM_2022_accepted_papers.html', conference_name)

In [34]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,STORM-GAN: Spatio-Temporal Meta-GAN for Cross-...,"Han Bao, Xun Zhou, Yiqun Xie, Yanhua Li, Xiaow...",https://doi.org/10.1109/ICDM54844.2022.00010,None,ICDM 2022
1,Which Companies are Likely to Invest: Knowledg...,"Chenyang Bu, Jiawei Zhang, Xingchen Yu, Le Wu,...",https://doi.org/10.1109/ICDM54844.2022.00011,None,ICDM 2022
2,TD3 with Reverse KL Regularizer for Offline Re...,"Yuanying Cai, Chuheng Zhang, Li Zhao, Wei Shen...",https://doi.org/10.1109/ICDM54844.2022.00012,None,ICDM 2022
3,Federated Fingerprint Learning with Heterogene...,"Tianshi Che, Zijie Zhang, Yang Zhou, Xin Zhao,...",https://doi.org/10.1109/ICDM54844.2022.00013,None,ICDM 2022
4,Robust Structure-aware Semi-supervised Learning.,Xu Chen,https://doi.org/10.1109/ICDM54844.2022.00014,None,ICDM 2022


In [35]:
save_to_database(df_papers, conference_name, DB_PATH)

171개의 논문이 ICDM 2022에 저장되었습니다.


# 2021

In [36]:
url = 'https://dblp.org/db/conf/icdm/icdm2021.html'
DB_PATH = "con_db/ICDM_conference_2021.db"
conference_name = 'ICDM 2021'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [37]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/ICDM_2021_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [38]:
df_papers = get_www_papers('html/ICDM_2021_accepted_papers.html', conference_name)

In [39]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Gated Information Bottleneck for Generalizatio...,"Francesco Alesiani, Shujian Yu, Xi Yu",https://doi.org/10.1109/ICDM51629.2021.00010,None,ICDM 2021
1,Partial Differential Equation Driven Dynamic G...,"Tianshu Bao, Xiaowei Jia, Jacob Zwart, Jeffrey...",https://doi.org/10.1109/ICDM51629.2021.00011,None,ICDM 2021
2,A Linear Primal-Dual Multi-Instance SVM for Bi...,"Lodewijk Brand, Lauren Zoe Baker, Carla Ellefs...",https://doi.org/10.1109/ICDM51629.2021.00012,None,ICDM 2021
3,Spatially and Robustly Hybrid Mixture Regressi...,"Wennan Chang, Pengdao Dang, Changlin Wan, Xiao...",https://doi.org/10.1109/ICDM51629.2021.00013,None,ICDM 2021
4,Differentially Private String Sanitization for...,"Huiping Chen, Changyu Dong, Liyue Fan, Grigori...",https://doi.org/10.1109/ICDM51629.2021.00014,None,ICDM 2021


In [40]:
save_to_database(df_papers, conference_name, DB_PATH)

197개의 논문이 ICDM 2021에 저장되었습니다.


# 2020

In [41]:
url = 'https://dblp.org/db/conf/icdm/icdm2020.html'
DB_PATH = "con_db/ICDM_conference_2020.db"
conference_name = 'ICDM 2020'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [42]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/ICDM_2020_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [43]:
df_papers = get_www_papers('html/ICDM_2020_accepted_papers.html', conference_name)

In [44]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Approximation algorithms for probabilistic $k$...,Sharareh Alipour,https://doi.org/10.1109/ICDM50108.2020.00009,None,ICDM 2020
1,Fast Spatial Autocorrelation.,"Anar Amgalan, Lilianne R. Mujica-Parodi, Steve...",https://doi.org/10.1109/ICDM50108.2020.00010,None,ICDM 2020
2,ViVA: Semi-supervised Visualization via Variat...,"Sungtae An, Shenda Hong, Jimeng Sun",https://doi.org/10.1109/ICDM50108.2020.00011,None,ICDM 2020
3,Defending Water Treatment Networks: Exploiting...,"Dongjie Wang, Pengyang Wang, Jingbo Zhou, Leil...",https://doi.org/10.1109/ICDM50108.2020.00012,None,ICDM 2020
4,Quality meets Diversity: A Model-Agnostic Fram...,"Haoyang Bi, Haiping Ma, Zhenya Huang, Yu Yin, ...",https://doi.org/10.1109/ICDM50108.2020.00013,None,ICDM 2020


In [45]:
save_to_database(df_papers, conference_name, DB_PATH)

182개의 논문이 ICDM 2020에 저장되었습니다.


# 2019

In [46]:
url = 'https://dblp.org/db/conf/icdm/icdm2019.html'
DB_PATH = "con_db/ICDM_conference_2019.db"
conference_name = 'ICDM 2019'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [47]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/ICDM_2019_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [48]:
df_papers = get_www_papers('html/ICDM_2019_accepted_papers.html', conference_name)

In [49]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Collaborative Graph Walk for Semi-Supervised M...,"Uchenna Akujuobi, Yufei Han, Qiannan Zhang, Xi...",https://doi.org/10.1109/ICDM.2019.00010,None,ICDM 2019
1,Dataset Recommendation via Variational Graph A...,"Basmah Altaf, Uchenna Akujuobi, Lu Yu, Xiangli...",https://doi.org/10.1109/ICDM.2019.00011,None,ICDM 2019
2,CUDA: Contradistinguisher for Unsupervised Dom...,"Sourabh Balgi, Ambedkar Dukkipati",https://doi.org/10.1109/ICDM.2019.00012,None,ICDM 2019
3,An Efficient Policy Gradient Method for Condit...,"Lei Cai, Shuiwang Ji",https://doi.org/10.1109/ICDM.2019.00013,None,ICDM 2019
4,Supervised Class Distribution Learning for GAN...,"Zixin Cai, Xinyue Wang, Mingjie Zhou, Jian Xu,...",https://doi.org/10.1109/ICDM.2019.00014,None,ICDM 2019


In [50]:
save_to_database(df_papers, conference_name, DB_PATH)

196개의 논문이 ICDM 2019에 저장되었습니다.


# 2018

In [51]:
url = 'https://dblp.org/db/conf/icdm/icdm2018.html'
DB_PATH = "con_db/ICDM_conference_2018.db"
conference_name = 'ICDM 2018'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [52]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/ICDM_2018_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [53]:
df_papers = get_www_papers('html/ICDM_2018_accepted_papers.html', conference_name)

In [54]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,On Big Wisdom.,"Minghui Wu, Xindong Wu",https://doi.org/10.1109/ICDM.2018.00009,None,ICDM 2018
1,Landscape of Practical Blockchain Systems and ...,C. Mohan,https://doi.org/10.1109/ICDM.2018.00010,None,ICDM 2018
2,Automatic Optical Coherence Tomography Imaging...,Ramamohanarao Kotagiri,https://doi.org/10.1109/ICDM.2018.00011,None,ICDM 2018
3,Blockchain Data Analytics.,"Cuneyt Gurcan Akcora, Murat Kantarcioglu, Yuli...",https://doi.org/10.1109/ICDM.2018.00013,None,ICDM 2018
4,Discourse Processing and Its Applications in T...,"Shafiq R. Joty, Giuseppe Carenini, Raymond T. ...",https://doi.org/10.1109/ICDM.2018.00014,None,ICDM 2018


In [55]:
save_to_database(df_papers, conference_name, DB_PATH)

196개의 논문이 ICDM 2018에 저장되었습니다.


# 2017

In [56]:
url = 'https://dblp.org/db/conf/icdm/icdm2017.html'
DB_PATH = "con_db/ICDM_conference_2017.db"
conference_name = 'ICDM 2017'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [57]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/ICDM_2017_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [58]:
df_papers = get_www_papers('html/ICDM_2017_accepted_papers.html', conference_name)

In [59]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Split Miner: Discovering Accurate and Simple B...,"Adriano Augusto, Raffaele Conforti, Marlon Dum...",https://doi.org/10.1109/ICDM.2017.9,None,ICDM 2017
1,A Deep Transfer Learning Approach for Improved...,"Debrup Banerjee, Kazi Aminul Islam, Gang Mei, ...",https://doi.org/10.1109/ICDM.2017.10,None,ICDM 2017
2,Many Heads are Better than One: Local Communit...,"Yuchen Bian, Jingchao Ni, Wei Cheng, Xiang Zhang",https://doi.org/10.1109/ICDM.2017.11,None,ICDM 2017
3,Knowledge Guided Short-Text Classification for...,"Shilei Cao, Buyue Qian, Changchang Yin, Xiaoyu...",https://doi.org/10.1109/ICDM.2017.12,None,ICDM 2017
4,A Generic Framework for Interesting Subspace C...,"Feng Chen, Baojian Zhou, Adil Alim, Liang Zhao",https://doi.org/10.1109/ICDM.2017.13,None,ICDM 2017


In [60]:
save_to_database(df_papers, conference_name, DB_PATH)

156개의 논문이 ICDM 2017에 저장되었습니다.


# 2016

In [61]:
url = 'https://dblp.org/db/conf/icdm/icdm2016.html'
DB_PATH = "con_db/ICDM_conference_2016.db"
conference_name = 'ICDM 2016'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [62]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/ICDM_2016_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [63]:
df_papers = get_www_papers('html/ICDM_2016_accepted_papers.html', conference_name)

In [64]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Auditing Black-Box Models for Indirect Influence.,"Philip Adler, Casey Falk, Sorelle A. Friedler,...",https://doi.org/10.1109/ICDM.2016.0011,None,ICDM 2016
1,Asynchronous Multi-task Learning.,"Inci M. Baytas, Ming Yan, Anil K. Jain, Jiayu ...",https://doi.org/10.1109/ICDM.2016.0012,None,ICDM 2016
2,Unsupervised Exceptional Attributed Sub-Graph ...,"Ahmed Anes Bendimerad, Marc Plantevit, Céline ...",https://doi.org/10.1109/ICDM.2016.0013,None,ICDM 2016
3,ADAGIO: Fast Data-Aware Near-Isometric Linear ...,"Jaroslaw Blasiok, Charalampos E. Tsourakakis",https://doi.org/10.1109/ICDM.2016.0014,None,ICDM 2016
4,Causal Inference by Compression.,"Kailash Budhathoki, Jilles Vreeken",https://doi.org/10.1109/ICDM.2016.0015,None,ICDM 2016


In [65]:
save_to_database(df_papers, conference_name, DB_PATH)

178개의 논문이 ICDM 2016에 저장되었습니다.


# 2015

In [66]:
url = 'https://dblp.org/db/conf/icdm/icdm2015.html'
DB_PATH = "con_db/ICDM_conference_2015.db"
conference_name = 'ICDM 2015'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [67]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/ICDM_2015_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [68]:
df_papers = get_www_papers('html/ICDM_2015_accepted_papers.html', conference_name)

In [69]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Efficient Graphlet Counting for Large Networks.,"Nesreen K. Ahmed, Jennifer Neville, Ryan A. Ro...",https://doi.org/10.1109/ICDM.2015.141,None,ICDM 2015
1,Diamond Sampling for Approximate Maximum All-P...,"Grey Ballard, Tamara G. Kolda, Ali Pinar, C. S...",https://doi.org/10.1109/ICDM.2015.46,None,ICDM 2015
2,Information Source Detection via Maximum A Pos...,"Biao Chang, Feida Zhu, Enhong Chen, Qi Liu",https://doi.org/10.1109/ICDM.2015.116,None,ICDM 2015
3,Influential Sustainability on Social Networks.,"Chien-Wei Chang, Po-An Yang, Ming-Han Lyu, Kun...",https://doi.org/10.1109/ICDM.2015.129,None,ICDM 2015
4,Towards Frequent Subgraph Mining on Single Lar...,"Yifan Chen, Xiang Zhao, Xuemin Lin, Yang Wang",https://doi.org/10.1109/ICDM.2015.88,None,ICDM 2015


In [70]:
save_to_database(df_papers, conference_name, DB_PATH)

146개의 논문이 ICDM 2015에 저장되었습니다.


# 2014

In [71]:
url = 'https://dblp.org/db/conf/icdm/icdm2014.html'
DB_PATH = "con_db/ICDM_conference_2014.db"
conference_name = 'ICDM 2014'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [72]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/ICDM_2014_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [73]:
df_papers = get_www_papers('html/ICDM_2014_accepted_papers.html', conference_name)

In [74]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Discriminative Learning on Exemplary Patterns ...,"Shin Ando, Einoshin Suzuki",https://doi.org/10.1109/ICDM.2014.122,None,ICDM 2014
1,Orthogonal Matching Pursuit for Sparse Quantil...,"Aleksandr Y. Aravkin, Aurélie C. Lozano, Ronny...",https://doi.org/10.1109/ICDM.2014.134,None,ICDM 2014
2,Quick Detection of High-Degree Entities in Lar...,"Konstantin Avrachenkov, Nelly Litvak, Liudmila...",https://doi.org/10.1109/ICDM.2014.95,None,ICDM 2014
3,Inferring Uncertain Trajectories from Partial ...,"Prithu Banerjee, Sayan Ranu, Sriram Raghavan",https://doi.org/10.1109/ICDM.2014.41,None,ICDM 2014
4,Tensor-Based Multi-view Feature Selection with...,"Bokai Cao, Lifang He, Xiangnan Kong, Philip S....",https://doi.org/10.1109/ICDM.2014.26,None,ICDM 2014


In [75]:
save_to_database(df_papers, conference_name, DB_PATH)

143개의 논문이 ICDM 2014에 저장되었습니다.
